# Decoding from PCA trajectories

Companion to the main tutorial. Loads the artifacts produced by
`tutorial_pca_trajectories_eegbci.ipynb` and runs the seven
trajectory-based decoding showcases of paper §7.6:

1. Decode condition from trial-mean PC scores (`Experiment`).
2. Trial-mean vs per-timepoint feature shapes.
3. Time-resolved decoding sweep (`TemporalDecoderConfig`).
4. Temporal generalisation matrix (King & Dehaene 2014).
5. Cross-subject CV via `group_kfold`.
6. Permutation null + statistical assessment.
7. Feature importance per PC (bonus).

**Prerequisite:** run the main tutorial first. Loading the artifacts
requires the directory `outputs/tutorial_eegbci/artifacts/` to exist.


In [ ]:
import sys
from pathlib import Path

_PROJECT_ROOT = Path.cwd()
if (_PROJECT_ROOT / 'coco_pipe').is_dir() is False and (_PROJECT_ROOT.parent / 'coco_pipe').is_dir():
    _PROJECT_ROOT = _PROJECT_ROOT.parent
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

import numpy as np
import pandas as pd

from coco_pipe.decoding import Experiment, ExperimentConfig
from coco_pipe.decoding.configs import (
    ClassicalModelConfig,
    CVConfig,
    FeatureSelectionConfig,
    LogisticRegressionConfig,
    StatisticalAssessmentConfig,
    TemporalDecoderConfig,
)
from coco_pipe.report import Report, Section, PlotlyElement
from coco_pipe.report.api import from_experiment_result
from coco_pipe.report.elements import InteractiveTableElement
from coco_pipe.viz.interactive import (
    plot_decoding_scores,
    plot_feature_importance,
    plot_fold_score_dispersion,
    plot_null_interval_summary,
    plot_subject_diagnostics,
    plot_temporal_generalization_matrix,
    plot_temporal_score_curve,
    plot_temporal_statistical_assessment,
)
from tutorials._helpers import LABEL_NAMES, load_artifacts

rng = np.random.default_rng(42)
ARTIFACTS_DIR = Path('outputs/tutorial_eegbci/artifacts')
OUT = Path('outputs/tutorial_eegbci_decoding')
OUT.mkdir(parents=True, exist_ok=True)

art = load_artifacts(ARTIFACTS_DIR, method='PCA')
container = art.trajectory_container
trial_labels = np.asarray(container.y).astype(int)
times = np.asarray(container.coords['time'])
subjects = np.asarray(container.coords.get('subject', container.ids))
print('container shape:', container.X.shape)

report = Report(title='Decoding from PCA trajectories', asset_urls='inline')


## 1 — Decode condition from trial-mean PC scores

One-vs-rest logistic regression, group-k-fold over subjects, ROC AUC.
Establishes whether the separation observed in the main tutorial
carries decodable information.


In [ ]:
trial_mean = container.X.mean(axis=1)[:, :3]  # (n_trials, 3)
config = ExperimentConfig(
    task='classification',
    models={'logreg': LogisticRegressionConfig(max_iter=500)},
    metrics=['accuracy', 'roc_auc'],
    cv=CVConfig(strategy='group_kfold', n_splits=5),
    feature_selection=FeatureSelectionConfig(enabled=False),
    n_jobs=1,
)
# When `groups` is provided without `sample_metadata`, Experiment.run
# auto-treats groups as Subject and synthesises a default Session.
result1 = Experiment(config).run(
    trial_mean, trial_labels, groups=subjects,
    feature_names=['PC1', 'PC2', 'PC3'],
)
summary1 = result1.summary()
print(summary1)

sec1 = Section('1 — Trial-mean decoding', icon='1')
sec1.add_element(InteractiveTableElement(summary1, title='Trial-mean decoding scores'))
report.add_section(sec1)


## 2 — Trial-mean vs per-timepoint features

Does adding temporal info beat the trial-mean? Flatten (PC × time)
and rerun the same `Experiment`.


In [ ]:
flat = container.X[:, :, :3].reshape(container.X.shape[0], -1)
result2 = Experiment(config).run(flat, trial_labels, groups=subjects)
summary2 = result2.summary()
comparison = pd.concat([summary1.assign(features='trial_mean'),
                        summary2.assign(features='flattened_time')])
print(comparison)

sec2 = Section('2 — Trial-mean vs flattened time', icon='2')
sec2.add_element(InteractiveTableElement(comparison, title='Feature-shape comparison'))
report.add_section(sec2)


## 3 — Time-resolved decoding (sliding estimator)

`TemporalDecoderConfig` wraps an MNE-style sliding estimator. The
decoding peak should align with the separation peak from step 8 of
the main tutorial — that overlay is the load-bearing decoding panel
for paper §7.6.


In [ ]:
temporal_config = ExperimentConfig(
    task='classification',
    models={
        'sliding': TemporalDecoderConfig(
            wrapper='sliding',
            base=ClassicalModelConfig(estimator='LogisticRegression', params={'max_iter': 500}),
        )
    },
    metrics=['accuracy'],
    cv=CVConfig(strategy='group_kfold', n_splits=5),
    n_jobs=1,
)
# MNE-style sliding estimator expects (n_trials, n_features, n_times)
X_time = container.X[:, :, :3].transpose(0, 2, 1)
result3 = Experiment(temporal_config).run(
    X_time, trial_labels, groups=subjects, time_axis=times,
)

# Single-model results: `model=` is auto-detected.
fig_temporal = plot_temporal_score_curve(result3, metric='accuracy')

sec3 = Section('3 — Time-resolved decoding', icon='3')
sec3.add_element(PlotlyElement(fig_temporal, height='420px'))
report.add_section(sec3)


## 4 — Temporal generalisation matrix

Train at time t, test at time t' (King & Dehaene 2014). Diagonal-
only patterns reflect transient codes; off-diagonal patterns reflect
stable / attractor-like representations (paper §3.2 last row).


In [ ]:
gen_config = ExperimentConfig(
    task='classification',
    models={
        'generalizing': TemporalDecoderConfig(
            wrapper='generalizing',
            base=ClassicalModelConfig(estimator='LogisticRegression', params={'max_iter': 500}),
        )
    },
    metrics=['accuracy'],
    cv=CVConfig(strategy='group_kfold', n_splits=3),
    n_jobs=1,
)
result4 = Experiment(gen_config).run(
    X_time, trial_labels, groups=subjects, time_axis=times,
)
fig_genmat = plot_temporal_generalization_matrix(result4, metric='accuracy')

sec4 = Section('4 — Temporal generalisation', icon='4')
sec4.add_element(PlotlyElement(fig_genmat, height='520px'))
report.add_section(sec4)


## 5 — Cross-subject generalisation

Per-fold ROC AUC dispersion + per-subject diagnostics. Shows whether
the trajectory representation transfers across subjects.


In [ ]:
fig_fold = plot_fold_score_dispersion(result1, metric='roc_auc')
fig_subj = plot_subject_diagnostics(result1)
sec5 = Section('5 — Cross-subject generalisation', icon='5')
sec5.add_element(PlotlyElement(fig_fold, height='420px'))
sec5.add_element(PlotlyElement(fig_subj, height='420px'))
report.add_section(sec5)


## 6 — Statistical assessment

Permutation null + multiple-comparison correction. Uses
`run_statistical_assessment` so the null is computed in one call
rather than a hand-rolled loop.


In [ ]:
sec6 = Section('6 — Statistical assessment', icon='6')
try:
    stat_config = ExperimentConfig(
        task='classification',
        models={'logreg': LogisticRegressionConfig(max_iter=500)},
        metrics=['accuracy'],
        cv=CVConfig(strategy='group_kfold', n_splits=5),
        statistical_assessment=StatisticalAssessmentConfig(
            enabled=True, random_state=42,
        ),
        n_jobs=1,
    )
    result6 = Experiment(stat_config).run(
        trial_mean, trial_labels, groups=subjects,
    )
    fig_null = plot_null_interval_summary(
        result6, model='logreg', metric='accuracy',
    )
    sec6.add_element(PlotlyElement(fig_null, height='420px'))
except (KeyError, ValueError) as exc:
    sec6.add_markdown(
        f'Statistical assessment failed: `{exc}`. '
        'Statistical assessment requires more samples / subjects '
        'than the 5-subject tutorial slice provides; the paper run '
        'on 109 subjects (`analysis/analysis_paper_eegbci_109.py`) '
        'produces a non-degenerate null.'
    )
report.add_section(sec6)


## 7 — Feature importance per PC (bonus)

Which PCs carried the decoding signal? `plot_feature_importance`
+ `plot_feature_stability` close the interpretability loop.


In [ ]:
sec7 = Section('7 — Feature importance', icon='7')
try:
    # `plot_feature_importance` wants feature->score data;
    # build it from the decoding result's feature scores helper.
    feat_scores = result1.get_feature_importances()
    if feat_scores is None or (hasattr(feat_scores, 'empty') and feat_scores.empty):
        raise ValueError('no feature importance data')
    fig_imp = plot_feature_importance(feat_scores, title='Feature importance per PC')
    sec7.add_element(PlotlyElement(fig_imp, height='420px'))
except (AttributeError, ValueError, TypeError) as exc:
    sec7.add_markdown(
        f'Feature importance plot unavailable: `{exc}`.'
        ' Logistic regression coefficients can be inspected directly via'
        ' `result1.get_predictions()` instead.'
    )
report.add_section(sec7)

report.save(str(OUT / 'report_tutorial_decoding.html'))
print('saved:', OUT / 'report_tutorial_decoding.html')
